# Training Notebooks

- [Vesuvius Surface 3D Detection in Keras-JAX](https://www.kaggle.com/code/ipythonx/vesuvius-surface-3d-detection-in-jax)
- [Vesuvius Surface 3D Detection in PyTorch](https://www.kaggle.com/code/ipythonx/vesuvius-surface-3d-detection-in-pytorch)
- [Vesuvius Surface 3D Detection in PyTorch Lightning](https://www.kaggle.com/code/ipythonx/train-vesuvius-surface-3d-detection-in-lightning)
- [[WIP] Vesuvius Surface 2.5D Detection](https://www.kaggle.com/code/ipythonx/wip-vesuvius-surface-2-5d-detection)

**Note**
1. The inference code below is adapted from the **Keras-JAX** version. The PyTorch and Lightning implementations follow the same workflow. Training was performed on a single Tesla T4 (16 GB VRAM) with extended epochs.
2. Both the training and inference pipelines are implemented using [`medicai`](https://github.com/innat/medic-ai), a **Keras 3** based multi-backend medical ML library designed for 2D and 3D classification and segmentation tasks. However, please note, `medicai` project is still new and actively evolving.

Entendido. He analizado a fondo el notebook `inference-baseline-transunet-lb-0-537.ipynb`. Este proyecto está diseñado para la competencia **Vesuvius Challenge - Surface Detection**, donde el objetivo es detectar superficies (fragmentos de papiro) en tomografías computarizadas (CT) en 3D.

A continuación, presento el análisis detallado de cada celda, el flujo lógico y la arquitectura técnica del proyecto.

## 1. Configuración de Entorno e Instalación (Celdas 1-2)

En un entorno de Kaggle sin internet, el autor utiliza archivos `.whl` locales para instalar dependencias críticas.

* **Celda 1:** Instalación de `keras_nightly` (versión 3.x), `medicai` (una librería especializada en segmentación médica 3D), `tifffile` e `imagecodecs`.
* **Celda 2 (Parche de Protobuf y Backend):** * Implementa un parche de compatibilidad para `google.protobuf` para asegurar que el método `GetPrototype` funcione con versiones más recientes.
* **Configuración Crítica:** Establece `os.environ["KERAS_BACKEND"] = "jax"`. Esto es fundamental, ya que Keras 3 permite alternar entre JAX, PyTorch y TensorFlow. JAX suele ser más rápido para inferencia en TPUs y GPUs de alto rendimiento.

## 2. Definición del Flujo de Datos (Celdas 3-5)

* **Celda 3 y 4:** Configura las rutas de entrada (`/test_images`) y salida (`/submission_masks`). Lee el archivo `test.csv` para identificar los volúmenes que deben ser procesados.
* **Celda 5 (Transformaciones):**
* Define `val_transformation`. Utiliza `ScaleIntensityRange` de la librería `medicai`.
* **Lógica:** Escala los valores de intensidad de los píxeles (originalmente de 0 a 255 en 8 bits) al rango . Esto es vital para que los pesos del modelo pre-entrenado procesen correctamente la distribución estadística de los datos.

## 3. Arquitectura del Modelo: TransUNet (Celdas 6-8)

El núcleo del proyecto es un modelo **TransUNet**, una arquitectura híbrida que combina la capacidad de localización de las UNets con la capacidad de atención global de los Transformers.

* **Estructura del Modelo:**
* **Encoder:** Utiliza un `seresnext50` (Squeeze-and-Excitation ResNeXt) adaptado para 3D.
* **Transformer:** Incorpora 12 capas de Vision Transformer (ViT) con 8 cabezales de atención (`num_heads`).
* **Dimensiones:** La entrada es de , lo que indica que procesa "cubos" o sub-volúmenes de la tomografía.


* **Carga de Pesos:** Carga un archivo `.h5` pre-entrenado en TPU. El modelo tiene aproximadamente **70 millones de parámetros**.

## 4. Inferencia por Ventana Deslizante (Celda 9)

Dado que las tomografías son demasiado grandes para la memoria de la GPU, se utiliza `SlidingWindowInference`.

* **`roi_size=(160, 160, 160)`:** El tamaño de la ventana.
* **`overlap=0.50`:** Las ventanas se solapan en un 50% para evitar artefactos en los bordes de cada cubo procesado.
* **`mode="gaussian"`:** Al promediar las predicciones del solapamiento, se da más peso al centro del cubo que a los bordes, mejorando la coherencia espacial.

## 5. Post-procesamiento y TTA (Celda 10)

Esta es la celda más compleja y rica en ingeniería de algoritmos. Implementa dos técnicas clave:

### A. Test Time Augmentation (TTA) por Rotación

La función `predict_probs_tta_rot` realiza 4 predicciones por cada volumen:

1. Imagen original ().
2. Rotación de  y  en el plano horizontal (HW).
3. Luego rota las predicciones de vuelta a su posición original y las promedia. Esto reduce drásticamente el ruido y mejora la robustez.

### B. Post-procesamiento Topológico (`topo_postprocess`)

Para limpiar la máscara segmentada, utiliza:

1. **Hysteresis Thresholding:** Usa dos umbrales (`T_low` y `T_high`). Si un píxel es "débil" (`> T_low`) pero está conectado a uno "fuerte" (`> T_high`), se mantiene. Esto preserva la conectividad de la superficie.
2. **3D Anisotropic Closing:** Una operación morfológica para cerrar pequeños huecos en el fragmento detectado, considerando que la resolución en el eje Z puede ser distinta al plano XY.
3. **Dust Removal:** Elimina objetos conectados que sean más pequeños que `dust_min_size`, descartando falsos positivos (ruido).

## 6. Generación de Submission (Celdas 11-13)

* El código itera sobre los archivos `.tif` del set de prueba.
* Carga el volumen, aplica la normalización, ejecuta la inferencia con TTA y el post-procesamiento.
* **Salida:** Los resultados se guardan en un archivo `.zip` que contiene las máscaras en formato `.tif`, cumpliendo con los requisitos de Kaggle.

---

### Resumen del Flujo Lógico:

1. **Carga** de sub-volumen 3D.
2. **Normalización** de intensidad .
3. **TTA:** Generar 4 rotaciones del volumen.
4. **Inferencia:** El TransUNet predice la probabilidad de "superficie" para cada rotación.
5. **Ensamble:** Promediar las 4 predicciones.
6. **Refinamiento:** Aplicar umbral de histéresis y limpieza morfológica 3D.
7. **Escritura:** Guardar máscara final.

Entendido. Como experto en **IA, Deep Learning, Matemáticas y Algoritmos**, profundizaré en los fundamentos matemáticos y las operaciones lógicas que rigen el funcionamiento de este notebook para la competencia **Vesuvius Challenge**.

Este proyecto no es una simple segmentación 2D; es un problema de visión por computadora en **espacio volumétrico 3D**. A continuación, desgloso la arquitectura, la técnica de inferencia y el post-procesamiento desde una perspectiva matemática rigurosa.

---

### 1. Arquitectura TransUNet: El Híbrido CNN-Transformer

El modelo cargado en la **Celda 6** es una **TransUNet**. Su diseño busca resolver la limitación de las CNNs puras (que tienen un campo receptivo local) mediante la integración de Transformers, que poseen una capacidad de atención global.

#### A. El Encoder (SEResNeXt50 3D)

El proceso comienza con una extracción de características jerárquica. La operación fundamental es la **convolución 3D**:



Donde  es el volumen de entrada y  es el núcleo (kernel). Al usar un **SEResNeXt**, se añade un bloque de **Squeeze-and-Excitation (SE)**, cuya matemática se basa en el recalibrado de canales:

1. **Squeeze:** Agregación de descriptores locales en un estadístico de canal mediante *Global Average Pooling*:


2. **Excitation:** Aplicación de un mecanismo de puerta (sigmoid) para capturar dependencias no lineales entre canales:



#### B. El Bottleneck de Transformer

La salida del encoder se "aplana" en una secuencia de parches para que el Transformer la procese. El núcleo matemático aquí es el **Multi-Head Self-Attention (MSA)**:



Donde:

*  son las matrices de **Query, Key y Value** obtenidas por proyecciones lineales del volumen.
*  es un factor de escala para evitar gradientes extremadamente pequeños en la función softmax.
* La función  normaliza las afinidades entre parches a un rango .

---

### 2. Inferencia por Ventana Deslizante (Gaussian Blending)

En la **Celda 9**, se configura `SlidingWindowInference` con `mode="gaussian"`. Matemáticamente, esto resuelve el problema de las discontinuidades en los bordes de los parches segmentados.

Cuando procesamos un volumen gigante dividiéndolo en cubos de  con un solapamiento (overlap) del 50%, cada voxel  del volumen original puede recibir múltiples predicciones . Para combinarlas, se aplica un peso basado en una **función Gaussiana multivariada** centrada en el cubo:



La predicción final para un voxel es la suma ponderada:

Esto asegura que las predicciones en los bordes de un parche (donde el modelo tiene menos contexto y suele fallar) tengan menos importancia que las predicciones centrales.

---

### 3. Post-procesamiento: Algoritmos de Refinamiento

La **Celda 10** contiene la lógica más sofisticada para limpiar la salida del modelo.

#### A. Umbral de Histéresis (3D Hysteresis)

A diferencia de un umbral simple (donde ), la histéresis utiliza dos umbrales:  y .

* **Puntos Fuertes:** .
* **Puntos Débiles:** .
El algoritmo define la máscara final  mediante un **crecimiento de regiones (Region Growing)** o propagación binaria:

Esto permite recuperar partes del fragmento de papiro que tienen baja confianza pero que están físicamente conectadas a una zona de alta confianza, eliminando al mismo tiempo el ruido aislado de alta intensidad.

#### B. Cierre Morfológico Anisotrópico

La función `build_anisotropic_struct` crea un elemento estructurante  (kernel binario). La operación de **Cierre (Closing)** se define como la Dilatación seguida de la Erosión:


1. **Dilatación ():** . Expande la máscara para rellenar huecos internos.
2. **Erosión ():** . Reduce la máscara para recuperar el tamaño original, pero manteniendo los huecos rellenados.
Al ser **anisotrópico**, el radio en el eje  () es distinto al plano  (), lo que indica que el modelo prioriza la continuidad entre capas de la tomografía.

#### C. Eliminación de Polvo (Dust Removal)

Matemáticamente, se basa en la identificación de **Componentes Conectados**. Sea  el conjunto de todos los objetos conectados en la máscara binaria. La operación elimina cualquier objeto  tal que su volumen (cardinalidad del conjunto de voxeles) sea menor a un umbral :

**Nota de tokens:** Esta respuesta ha cubierto la profundidad matemática de la arquitectura y el post-procesamiento. Si deseas un análisis línea por línea de la implementación del TTA (Test Time Augmentation) o de la normalización de tensores en JAX, avísame y continuaré en la siguiente respuesta.

# Inference

In [1]:
var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
    "$var"/keras_nightly-3.12.0.dev2025100703-py3-none-any.whl \
    "$var"/tifffile-2025.12.12-py3-none-any.whl \
    "$var"/imagecodecs-2025.11.11-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl \
    "$var"/medicai-0.0.3-py3-none-any.whl \
    --no-index \
    --find-links "$var"

Looking in links: /kaggle/input/vsdetection-packages-offline-installer-only/whls
Processing /kaggle/input/vsdetection-packages-offline-installer-only/whls/keras_nightly-3.12.0.dev2025100703-py3-none-any.whl
Processing /kaggle/input/vsdetection-packages-offline-installer-only/whls/tifffile-2025.12.12-py3-none-any.whl
Processing /kaggle/input/vsdetection-packages-offline-installer-only/whls/imagecodecs-2025.11.11-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/vsdetection-packages-offline-installer-only/whls/medicai-0.0.3-py3-none-any.whl
  Attempting uninstall: tifffile
    Found existing installation: tifffile 2025.6.11
    Uninstalling tifffile-2025.6.11:
      Successfully uninstalled tifffile-2025.6.11


In [2]:
# --- Protobuf compatibility patch (for old code using MessageFactory.GetPrototype) ---

try:
    from google.protobuf import message_factory as _message_factory

    # Only patch if the method is missing (protobuf >= 5)
    if not hasattr(_message_factory.MessageFactory, "GetPrototype"):
        from google.protobuf.message_factory import GetMessageClass

        def _GetPrototype(self, descriptor):
            # Old API used MessageFactory().GetPrototype(descriptor)
            # New API is GetMessageClass(descriptor). We just bridge them.
            return GetMessageClass(descriptor)

        _message_factory.MessageFactory.GetPrototype = _GetPrototype
        print("Patched protobuf: added MessageFactory.GetPrototype")
    else:
        print("protobuf already has MessageFactory.GetPrototype; no patch needed.")
except Exception as e:
    print("Could not patch protobuf MessageFactory:", e)

import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from medicai.transforms import (
    Compose,
    ScaleIntensityRange,
)
from medicai.models import SegFormer, TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import zipfile
import tifffile
from matplotlib import pyplot as plt

keras.config.backend(), keras.version()

protobuf already has MessageFactory.GetPrototype; no patch needed.


2025-12-31 18:59:16.169056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767207556.375552      27 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767207556.433700      27 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


('jax', '3.12.0.dev2025100703')

**Dataset**

In [3]:
root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
test_dir = f"{root_dir}/test_images"
output_dir = "/kaggle/working/submission_masks"
zip_path = "/kaggle/working/submission.zip"
os.makedirs(output_dir, exist_ok=True)

In [4]:
test_df = pd.read_csv(f"{root_dir}/test.csv")
test_df.head()

,id,scroll_id
0,1407735,26002


**Transformation**

In [5]:
def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        ScaleIntensityRange(
            keys=["image"],
            a_min = 0,
            a_max = 255,
            b_min = 0,
            b_max = 1,
            clip = True,
        ),
    ])
    result = pipeline(data)
    return result["image"]

**Model**

In [6]:
num_classes=3

def get_model():
    ## LB: 0.486
    # model = SegFormer(
    #     input_shape=(128, 128, 128, 1),
    #     encoder_name='mit_b2',
    #     classifier_activation='softmax',
    #     num_classes=num_classes,
    # )
    # model.load_weights(
    #     "/kaggle/input/vsd-model/keras/segformer.mit.b2/2/segformer.mit.b2.weights.h5"
    # )

    ## LB: 0.5 
    model = TransUNet(
        input_shape=(160, 160, 160, 1),
        encoder_name='seresnext50',
        classifier_activation='softmax',
        num_classes=num_classes,
    )
    model.load_weights(
        "/kaggle/input/train-vesuvius-surface-3d-detection-on-tpu/model.weights.h5"
    )
    return model

In [7]:
model = get_model()
model.count_params() / 1e6
# predictor = tf.function(model, jit_compile=True)


70.056598

In [8]:
model.instance_describe()

Instance of TransUNet
  • input_shape: (160, 160, 160, 1)
  • num_classes: 3
  • num_queries: 100
  • encoder: SEResNeXt50(
    • name: 'SEResNeXt503D'
    • trainable: True
    • input_shape: (160, 160, 160, 1)
    • include_rescaling: False
    )
  • encoder_name: 'seresnext50'
  • encoder_depth: 5
  • classifier_activation: softmax
  • num_vit_layers: 12
  • num_heads: 8
  • embed_dim: 512
  • mlp_dim: 1024
  • dropout_rate: 0.1
  • decoder_activation: leaky_relu
  • decoder_filters: (256, 128, 64, 32, 16)
  • encoder: SEResNeXt50(
    • name: 'SEResNeXt503D'
    • trainable: True
    • input_shape: (160, 160, 160, 1)
    • include_rescaling: False
    )

**Sliding Window Inference**

In [9]:
pred = SlidingWindowInference(
    model,
    roi_size=(160,160,160),
    num_classes = 3,
    mode="gaussian",
    overlap=0.50,
    sw_batch_size = 1
)



In [10]:
import numpy as np
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects


def load_volume(path):
    vol = tifffile.imread(path)          # (D, H, W)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]           # (1, D, H, W, 1)
    return vol


# ==========================================
# ROTATION TTA HELPERS (CLOCKWISE)
# ==========================================
def rot90_volume(vol, k):
    """
    Rotate volume k times 90° clockwise in HW plane.
    vol:
      (1, D, H, W, 1) OR (D, H, W)
    """
    if vol.ndim == 5:
        return np.rot90(vol, k=-k, axes=(2, 3))
    else:
        return np.rot90(vol, k=-k, axes=(1, 2))


def unrot90_volume(vol, k):
    return rot90_volume(vol, (4 - k) % 4)


def predict_probs_tta_rot(sample):
    """
    4x rotation TTA: 0°, 90°, 180°, 270°
    sample: (1, D, H, W, 1)
    returns: averaged probs (D, H, W)
    """
    probs_accum = []

    for k in range(4):
        s_rot = rot90_volume(sample, k)

        out = pred(s_rot)              # (1, D, H, W, 2)
        out = np.asarray(out)
        probs = out[0, ..., 1]         # (D, H, W)

        probs = unrot90_volume(probs, k)
        probs_accum.append(probs)

    return np.mean(probs_accum, axis=0)


# ==========================================
# HELPER: Anisotropic Structure Builder
# ==========================================
def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius

    if z == 0 and r == 0:
        return None

    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct

    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct

    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


# ==========================================
# MAIN POST-PROCESSING LOGIC
# ==========================================
def topo_postprocess(
    probs,          # (D, H, W)
    T_low=0.90,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    # --- Step 1: 3D Hysteresis ---
    strong = probs >= T_high
    weak   = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # --- Step 2: 3D Anisotropic Closing ---
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # --- Step 3: Dust Removal ---
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)


# ==========================================
# PREDICT (WITH ROTATION TTA)
# ==========================================
def predict(
    sample,
    iid=None,
    T_low=0.45,
    T_high=0.85,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    """
    sample: (1, D, H, W, 1)
    """

    # --------- ROTATION TTA PROBS ---------
    probs_fg = predict_probs_tta_rot(sample)   # (D, H, W)

    if iid is not None:
        np.save(iid, probs_fg)

    # --------- POSTPROCESS (UNCHANGED) ----
    final = topo_postprocess(
        probs_fg,
        T_low=T_low,
        T_high=T_high,
        z_radius=z_radius,
        xy_radius=xy_radius,
        dust_min_size=dust_min_size,
    )

    return final  # (D, H, W) uint8 {0,1}


**Prediction and Zip Submission**

In [11]:
testing = False

In [12]:
if testing:
    
    test_dir = "/kaggle/input/vesuvius-challenge-surface-detection/train_images"
    test_df = pd.read_csv(f"{root_dir}/train.csv")
    test_ids = {956073442, 961304774,969293709,975031774,985841575,992852942}
    test_df = (
        test_df
        .loc[test_df["id"].isin(test_ids)]
        .reset_index(drop=True)
    )

In [13]:
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for image_id in test_df["id"]:
        tif_path = f"{test_dir}/{image_id}.tif"
            
        volume = load_volume(tif_path)
        volume = val_transformation(volume)
        if testing :
            output = predict(volume,f"{image_id}") 
        else :
            output = predict(volume)
        
        out_path = f"{output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, output.astype(np.uint8))

        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)

print("Submission ZIP:", zip_path)

I0000 00:00:1767207585.146101      27 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3303 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
Total patch 27:   0%|          | 0/27 [00:00<?, ?it/s]2025-12-31 18:59:54.959451: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-31 18:59:55.149712: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
Total patch 27: 100%|██████████| 27/27 [00:12<00:00,  2.14it/s]


Submission ZIP: /kaggle/working/submission.zip
